# Week 4 — StayMetrics Bronze Ingestion

**Project:** StayMetrics: Hotel Revenue & Occupancy Hub  
**Team:** Team 08

## Objective

Load all approved StayMetrics batch source files from the Unity Catalog
Volume into persistent Bronze Delta tables.

Bronze preserves the original source business values and adds technical
ingestion and lineage metadata.

### Week 4 scope
- Read approved batch sources
- Inspect source data
- Create Bronze Delta tables
- Add ingestion metadata
- Reconcile source and Bronze row counts
- Verify metadata
- Test a controlled rerun
- Inspect Delta history

No Silver transformations, data cleansing, Gold aggregation, Power BI,
or streaming work is performed in Week 4.


In [0]:

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "default"

VOLUME = "/Volumes/workspace/default/my_files/staymetrics/working_data"

RUN_ID = "staymetrics_w04_run_001"
SCHEMA_VERSION = "staymetrics_bronze_v1"

print("Catalog:", CATALOG)
print("Schema:", SCHEMA)
print("Volume:", VOLUME)
print("Run ID:", RUN_ID)
print("Schema Version:", SCHEMA_VERSION)

In [0]:
print("===== APPROVED STAYMETRICS BATCH SOURCES =====")

sources = [
    ("bookings.parquet", "Parquet", "bronze_staymetrics_bookings"),
    ("guests.csv", "CSV", "bronze_staymetrics_guests"),
    ("rooms.json", "JSON", "bronze_staymetrics_rooms"),
    ("rate_plans.csv", "CSV", "bronze_staymetrics_rate_plans"),
    ("room_nights.csv", "CSV", "bronze_staymetrics_room_nights")
]

for filename, file_format, table_name in sources:
    print(f"{filename:25} | {file_format:8} | {table_name}")

print("\n===== VOLUME FILE CHECK =====")
display(dbutils.fs.ls(VOLUME))


In [0]:
# BOOKINGS — Source inspection

bookings_src = (
    spark.read
    .parquet(f"{VOLUME}/bookings.parquet")
)

bookings_source_count = bookings_src.count()

print("Source: bookings.parquet")
print("Format: Parquet")
print("Source rows:", bookings_source_count)

bookings_src.printSchema()
display(bookings_src.limit(5))


In [0]:
bookings_path = f"{VOLUME}/bookings.parquet"

bookings_src = (
    spark.read
    .parquet(bookings_path)
)

bookings_source_count = bookings_src.count()

print("Source: bookings.parquet")
print("Format: Parquet")
print("Source rows:", bookings_source_count)

bookings_src.printSchema()

display(bookings_src.limit(5))

In [0]:
booking_hash_columns = bookings_src.columns

bookings_bronze = (
    bookings_src
    .withColumn("_source_file_name", F.lit("bookings.parquet"))
    .withColumn("_source_file_path", F.lit(bookings_path))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit(""))
                    for c in booking_hash_columns
                ]
            ),
            256
        )
    )
)

bookings_bronze_table = f"{CATALOG}.{SCHEMA}.bronze_staymetrics_bookings"

(
    bookings_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bookings_bronze_table)
)

print("Bronze table created:", bookings_bronze_table)
print("Bronze rows:", spark.table(bookings_bronze_table).count())

display(
    spark.table(bookings_bronze_table).limit(5)
)

In [0]:
guests_path = f"{VOLUME}/guests.csv"

guests_src = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(guests_path)
)

guests_source_count = guests_src.count()

print("Source: guests.csv")
print("Format: CSV")
print("Source rows:", guests_source_count)

guests_src.printSchema()

display(guests_src.limit(5))

In [0]:
guest_hash_columns = guests_src.columns

guests_bronze = (
    guests_src
    .withColumn("_source_file_name", F.lit("guests.csv"))
    .withColumn("_source_file_path", F.lit(guests_path))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit(""))
                    for c in guest_hash_columns
                ]
            ),
            256
        )
    )
)

guests_bronze_table = f"{CATALOG}.{SCHEMA}.bronze_staymetrics_guests"

(
    guests_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(guests_bronze_table)
)

print("Bronze table created:", guests_bronze_table)
print("Bronze rows:", spark.table(guests_bronze_table).count())

display(
    spark.table(guests_bronze_table).limit(5)
)

In [0]:
rooms_path = f"{VOLUME}/rooms.json"

rooms_src = (
    spark.read
    .option("multiLine", True)
    .json(rooms_path)
)

rooms_source_count = rooms_src.count()

print("Source: rooms.json")
print("Format: JSON")
print("Source rows:", rooms_source_count)

rooms_src.printSchema()

display(rooms_src.limit(5))

In [0]:
# ROOMS — Create Bronze table

room_hash_columns = rooms_src.columns

rooms_bronze = (
    rooms_src
    .withColumn("_source_file_name", F.lit("rooms.json"))
    .withColumn("_source_file_path", F.lit(rooms_path))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(c).cast("string"),
                        F.lit("")
                    )
                    for c in room_hash_columns
                ]
            ),
            256
        )
    )
)

rooms_bronze_table = f"{CATALOG}.{SCHEMA}.bronze_staymetrics_rooms"

rooms_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(rooms_bronze_table)

print("Bronze table created:", rooms_bronze_table)
print("Bronze rows:", spark.table(rooms_bronze_table).count())

display(
    spark.table(rooms_bronze_table).limit(5)
)

In [0]:
# RATE PLANS + ROOM NIGHTS — Bronze ingestion

# ---------------- RATE PLANS ----------------

rate_plans_path = f"{VOLUME}/rate_plans.csv"

rate_plans_src = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(rate_plans_path)
)

rate_plans_source_count = rate_plans_src.count()

rate_plan_hash_columns = rate_plans_src.columns

rate_plans_bronze = (
    rate_plans_src
    .withColumn("_source_file_name", F.lit("rate_plans.csv"))
    .withColumn("_source_file_path", F.lit(rate_plans_path))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit(""))
                    for c in rate_plan_hash_columns
                ]
            ),
            256
        )
    )
)

rate_plans_bronze_table = (
    f"{CATALOG}.{SCHEMA}.bronze_staymetrics_rate_plans"
)

rate_plans_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(rate_plans_bronze_table)


# ---------------- ROOM NIGHTS ----------------

room_nights_path = f"{VOLUME}/room_nights.csv"

room_nights_src = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(room_nights_path)
)

room_nights_source_count = room_nights_src.count()

room_night_hash_columns = room_nights_src.columns

room_nights_bronze = (
    room_nights_src
    .withColumn("_source_file_name", F.lit("room_nights.csv"))
    .withColumn("_source_file_path", F.lit(room_nights_path))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit(""))
                    for c in room_night_hash_columns
                ]
            ),
            256
        )
    )
)

room_nights_bronze_table = (
    f"{CATALOG}.{SCHEMA}.bronze_staymetrics_room_nights"
)

room_nights_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(room_nights_bronze_table)


print("Rate Plans Bronze:", rate_plans_bronze_table)
print("Rate Plans rows:", spark.table(rate_plans_bronze_table).count())

print("Room Nights Bronze:", room_nights_bronze_table)
print("Room Nights rows:", spark.table(room_nights_bronze_table).count())

In [0]:
# FINAL WEEK 4 ACCEPTANCE CHECKS

bronze_tables = [
    bookings_bronze_table,
    guests_bronze_table,
    rooms_bronze_table,
    rate_plans_bronze_table,
    room_nights_bronze_table
]

# 1. Bronze table existence
existence = [
    (t, "PASS" if spark.catalog.tableExists(t) else "CHECK")
    for t in bronze_tables
]

print("===== BRONZE TABLE EXISTENCE =====")
display(
    spark.createDataFrame(
        existence,
        ["bronze_table", "status"]
    )
)

# 2. Metadata check
metadata_columns = [
    "_source_file_name",
    "_source_file_path",
    "_ingestion_timestamp",
    "_ingestion_run_id",
    "_schema_version",
    "_record_hash"
]

metadata_results = []

for table in bronze_tables:
    cols = spark.table(table).columns
    missing = [c for c in metadata_columns if c not in cols]

    metadata_results.append(
        (
            table,
            "PASS" if not missing else "CHECK",
            ", ".join(missing) if missing else "None"
        )
    )

print("===== METADATA CHECK =====")
display(
    spark.createDataFrame(
        metadata_results,
        ["bronze_table", "status", "missing_metadata"]
    )
)

# 3. Source vs Bronze reconciliation
reconciliation = [
    (
        "bookings.parquet",
        bookings_source_count,
        spark.table(bookings_bronze_table).count()
    ),
    (
        "guests.csv",
        guests_source_count,
        spark.table(guests_bronze_table).count()
    ),
    (
        "rooms.json",
        rooms_source_count,
        spark.table(rooms_bronze_table).count()
    ),
    (
        "rate_plans.csv",
        rate_plans_source_count,
        spark.table(rate_plans_bronze_table).count()
    ),
    (
        "room_nights.csv",
        room_nights_source_count,
        spark.table(room_nights_bronze_table).count()
    )
]

reconciliation_df = (
    spark.createDataFrame(
        reconciliation,
        ["source_file", "source_rows", "bronze_rows"]
    )
    .withColumn(
        "status",
        F.when(
            F.col("source_rows") == F.col("bronze_rows"),
            "MATCH"
        ).otherwise("CHECK")
    )
)

print("===== SOURCE VS BRONZE RECONCILIATION =====")
display(reconciliation_df)

In [0]:
# CONTROLLED RERUN + DELTA HISTORY

print("===== CONTROLLED RERUN =====")

(
    bookings_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bookings_bronze_table)
)

after_rerun = spark.table(bookings_bronze_table).count()

print("Original source rows:", bookings_source_count)
print("Bronze rows after rerun:", after_rerun)

print(
    "RERUN STATUS:",
    "PASS" if after_rerun == bookings_source_count else "CHECK"
)

print("\n===== DELTA HISTORY =====")

display(
    spark.sql(
        f"DESCRIBE HISTORY {bookings_bronze_table}"
    ).select(
        "version",
        "timestamp",
        "operation"
    ).limit(5)
)

In [0]:
print("===== WEEK 4 COMPLETE =====")

final_tables = [
    ("Bookings", bookings_bronze_table),
    ("Guests", guests_bronze_table),
    ("Rooms", rooms_bronze_table),
    ("Rate Plans", rate_plans_bronze_table),
    ("Room Nights", room_nights_bronze_table)
]

results = []

for name, table in final_tables:
    results.append(
        (
            name,
            table,
            spark.table(table).count()
        )
    )

display(
    spark.createDataFrame(
        results,
        ["source", "bronze_table", "rows"]
    )
)

print("Week 4 Bronze ingestion completed.")

## Acceptance Criteria

- Bronze table exists
- Metadata columns exist
- Raw count and Bronze count reconcile
- Screenshot saved in `screenshots/`
